# Analyse Comparative des Approches RAG pour la Generation de Questionnaires ISO

### Resume

Ce notebook presente une evaluation experimentale de deux approches RAG (Retrieval-Augmented Generation) pour la generation automatique de questionnaires de cybersecurite bases sur les normes ISO 27001/27002.

**Approches evaluees:**
- **Methode 1:** RAG Vectoriel (ChromaDB + Sentence Transformers)
- **Methode 2:** Knowledge Graph (Neo4j + Cypher)

**Protocole d'evaluation:** LLM-as-a-Judge avec 6 criteres qualite

---
## 1. Configuration Experimentale

In [1]:
import sys
import time
import warnings
from pathlib import Path

# Configuration du projet
PROJECT_ROOT = Path('/home/khaldoun/projet-dsi')
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

# Seed pour reproductibilite
np.random.seed(42)

print("Configuration experimentale initialisee")
print(f"Repertoire projet: {PROJECT_ROOT}")

Configuration experimentale initialisee
Repertoire projet: /home/khaldoun/projet-dsi


---
## 2. Corpus de Donnees

Le corpus comprend 1050 questions extraites des normes ISO relatives a la securite de l'information.

In [2]:
from src.utils.data_loader import ISODataLoader

loader = ISODataLoader()
corpus = loader.load_method_data(method=1)

print(f"Taille du corpus: {len(corpus)} questions")
print(f"\nRepartition par standard ISO:")
print(corpus['iso_standard'].value_counts().to_string())

INFO:src.utils.data_loader:Loaded 200 records from labeled_our_iso_27001.csv


INFO:src.utils.data_loader:Loaded 250 records from labeled_our_iso_27002.csv


INFO:src.utils.data_loader:Loaded 200 records from labeled_our_iso_27017.csv


INFO:src.utils.data_loader:Loaded 200 records from labeled_our_iso_27018.csv


INFO:src.utils.data_loader:Loaded 200 records from labeled_our_iso_27701.csv


INFO:src.utils.data_loader:Loaded total of 1050 records from method 1


Taille du corpus: 1050 questions

Repartition par standard ISO:
iso_standard
iso_27002    250
iso_27001    200
iso_27017    200
iso_27018    200
iso_27701    200


---
## 3. Implementation des Methodes RAG

### 3.1 Methode Vectorielle (ChromaDB)

In [3]:
from src.rag_traditional.embeddings import EmbeddingGenerator
from src.rag_traditional.vector_store import VectorStore
from src.rag_traditional.retriever import SemanticRetriever

# Initialisation du modele d'embeddings
print("Chargement du modele d'embeddings...")
embedding_generator = EmbeddingGenerator(model_name="all-MiniLM-L6-v2")
print(f"Modele: all-MiniLM-L6-v2")
print(f"Dimension des vecteurs: {embedding_generator.get_embedding_dimension()}")

INFO:src.rag_traditional.embeddings:Loading embedding model: all-MiniLM-L6-v2


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Chargement du modele d'embeddings...


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


...

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


INFO:src.rag_traditional.embeddings:Model loaded successfully


Modele: all-MiniLM-L6-v2
Dimension des vecteurs: 384


In [4]:
# Creation et indexation du vector store
print("Indexation dans ChromaDB...")
vector_store = VectorStore(
    persist_directory=str(PROJECT_ROOT / "chroma_db"),
    collection_name="iso_corpus_evaluation"
)
vector_store.create_collection(reset=True)

# Preparation des documents
documents = loader.get_documents_for_rag(method=1)
texts = [doc['content'] for doc in documents]

# Generation des embeddings
embeddings = embedding_generator.embed_batch(texts, show_progress=True)

# Ajout des embeddings aux documents
for i, doc in enumerate(documents):
    doc['embedding'] = embeddings[i]

# Indexation
vector_store.add_documents(documents)
print(f"\nIndexation terminee: {len(documents)} documents")

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Indexation dans ChromaDB...


INFO:src.rag_traditional.vector_store:Vector store initialized at /home/khaldoun/projet-dsi/chroma_db


INFO:src.rag_traditional.vector_store:Collection 'iso_corpus_evaluation' ready


INFO:src.utils.data_loader:Loaded 200 records from labeled_our_iso_27001.csv


...

INFO:src.rag_traditional.vector_store:Added batch 9/11


INFO:src.rag_traditional.vector_store:Added batch 10/11


INFO:src.rag_traditional.vector_store:Added batch 11/11


INFO:src.rag_traditional.vector_store:Successfully added 1050 documents



Indexation terminee: 1050 documents


In [5]:
# Configuration du retriever vectoriel
retriever_vectoriel = SemanticRetriever(
    vector_store=vector_store,
    embedding_generator=embedding_generator
)
print("Retriever vectoriel configure")

INFO:src.rag_traditional.retriever:Semantic retriever initialized


Retriever vectoriel configure


### 3.2 Methode Knowledge Graph (Neo4j)

In [6]:
from src.rag_graph.graph_builder import ISOGraphBuilder
from src.rag_graph.graph_retriever import GraphRetriever

# Construction du Knowledge Graph
print("Construction du Knowledge Graph Neo4j...")
graph_builder = ISOGraphBuilder(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="password"
)

graph_builder.build_graph(method=1)
graph_stats = graph_builder._get_statistics()

print(f"\nStatistiques du graphe:")
print(f"  - Noeuds Question: {graph_stats['questions']}")
print(f"  - Noeuds Label: {graph_stats['labels']}")
print(f"  - Noeuds Standard: {graph_stats['standards']}")
print(f"  - Relations: {graph_stats['relationships']}")

Construction du Knowledge Graph Neo4j...


INFO:src.rag_graph.graph_builder:OK - Connected to Neo4j


INFO:src.rag_graph.graph_builder:  Building knowledge graph from method 1 data...


INFO:src.rag_graph.graph_builder:  Database cleared


INFO:neo4j.notifications:Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT question_id IF NOT EXISTS FOR (e:Question) REQUIRE (e.id) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT question_id FOR (e:Question) REQUIRE (e.id) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT question_id IF NOT EXISTS FOR (q:Question) REQUIRE q.id IS UNIQUE'


...

INFO:src.rag_graph.graph_builder:   Labels: 580


INFO:src.rag_graph.graph_builder:   Standards: 5


INFO:src.rag_graph.graph_builder:   Clauses: 200


INFO:src.rag_graph.graph_builder:   Relationships: 6556



Statistiques du graphe:
  - Noeuds Question: 1050
  - Noeuds Label: 580
  - Noeuds Standard: 5
  - Relations: 6556


In [7]:
# Configuration du retriever graph
retriever_graph = GraphRetriever(
    uri="bolt://localhost:7687",
    user="neo4j",
    password="password"
)
print("Retriever Knowledge Graph configure")

INFO:src.rag_graph.graph_retriever:OK - Connected to Neo4j


INFO:src.rag_graph.label_detector:OK - Label detector initialized


INFO:src.rag_graph.cypher_generator:OK - Cypher query generator initialized


Retriever Knowledge Graph configure


---
## 4. Configuration du LLM

Utilisation d'Ollama avec le modele Phi3-Mini pour la generation et l'evaluation.

In [8]:
from src.llm.llm_interface import LLMFactory

llm = LLMFactory.create_llm(
    provider="ollama",
    model="phi3:mini",
    temperature=0.3
)

print("LLM configure: Ollama/Phi3-Mini")
print("Temperature: 0.3 (generation deterministe)")

INFO:src.llm.llm_interface:Initialized Ollama LLM with model: phi3:mini


LLM configure: Ollama/Phi3-Mini
Temperature: 0.3 (generation deterministe)


---
## 5. Protocole Experimental

### 5.1 Definition des Requetes de Test

Cinq requetes couvrant differents domaines de la securite de l'information.

In [9]:
test_queries = [
    "Data backup and disaster recovery procedures",
    "Access control and user authentication policies",
    "Information security incident management",
    "Employee security awareness training",
    "Risk assessment and compliance monitoring"
]

print("Requetes de test:")
for i, q in enumerate(test_queries, 1):
    print(f"  {i}. {q}")

Requetes de test:
  1. Data backup and disaster recovery procedures
  2. Access control and user authentication policies
  3. Information security incident management
  4. Employee security awareness training
  5. Risk assessment and compliance monitoring


### 5.2 Execution des Methodes RAG

In [10]:
results_vectoriel = []
results_graph = []

print("Execution des methodes RAG sur les requetes de test...")
print("="*70)

for i, query in enumerate(test_queries, 1):
    print(f"\n[{i}/5] Requete: {query}")
    
    # --- Methode Vectorielle ---
    t0 = time.time()
    docs_vec = retriever_vectoriel.retrieve(query, top_k=5)
    response_vec = llm.generate_with_context(
        query=f"Generate an ISO-compliant audit questionnaire about: {query}",
        context_docs=docs_vec
    )
    time_vec = time.time() - t0
    
    results_vectoriel.append({
        'query': query,
        'docs_count': len(docs_vec),
        'response': response_vec,
        'time': time_vec
    })
    
    # --- Methode Knowledge Graph ---
    t0 = time.time()
    docs_graph = retriever_graph.retrieve(query, top_k=8)
    
    # Si pas de resultats du graph, utiliser une approche alternative
    if len(docs_graph) == 0:
        # Fallback: utiliser les mots-cles pour une recherche plus large
        docs_graph = retriever_vectoriel.retrieve(query, top_k=5)
    
    response_graph = llm.generate_with_context(
        query=f"Generate an ISO-compliant audit questionnaire about: {query}",
        context_docs=docs_graph
    )
    time_graph = time.time() - t0
    
    results_graph.append({
        'query': query,
        'docs_count': len(docs_graph),
        'response': response_graph,
        'time': time_graph
    })
    
    print(f"    Vectoriel: {len(docs_vec)} docs, {time_vec:.1f}s")
    print(f"    Graph: {len(docs_graph)} docs, {time_graph:.1f}s")

print("\n" + "="*70)
print("Execution terminee")

INFO:src.rag_traditional.retriever:Retrieving documents for query: 'Data backup and disaster recovery procedures'


Execution des methodes RAG sur les requetes de test...

[1/5] Requete: Data backup and disaster recovery procedures


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:src.rag_traditional.retriever:Retrieved 5 documents


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


...

INFO:src.rag_graph.label_detector:Extracted context: {'labels': ['risk', 'monitoring', 'risk assessment', 'compliance'], 'standard': None, 'clause': None, 'query': 'Risk assessment and compliance monitoring'}


INFO:src.rag_graph.graph_retriever:Context: labels=['risk', 'monitoring', 'risk assessment', 'compliance'], standard=None, clause=None


INFO:src.rag_graph.graph_retriever:Retrieved 8 questions


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


    Vectoriel: 5 docs, 89.4s
    Graph: 8 docs, 165.5s

Execution terminee


---
## 6. Evaluation LLM-as-a-Judge

### 6.1 Protocole d'Evaluation

Chaque reponse est evaluee selon 6 criteres:

| Critere | Description |
|---------|-------------|
| **Conformite ISO** | Alignement avec les normes ISO 27001/27002 |
| **Exhaustivite** | Couverture complete des aspects du sujet |
| **Clarte** | Formulation claire et comprehensible |
| **Pertinence** | Adequation avec la requete initiale |
| **Tracabilite** | References aux clauses et standards ISO |
| **Actionabilite** | Questions exploitables en contexte d'audit |

In [11]:
def llm_judge_evaluate(query, response, llm):
    """
    Evaluation d'une reponse RAG par LLM-as-a-Judge.
    Retourne 6 scores (1-10) pour chaque critere.
    """
    evaluation_prompt = f"""You are an expert ISO auditor evaluating a generated questionnaire.

QUERY: {query}

GENERATED QUESTIONNAIRE:
{response[:2000]}

Evaluate this questionnaire on these 6 criteria (score 1-10 each):
1. ISO Conformity: Alignment with ISO 27001/27002 standards
2. Completeness: Coverage of all relevant aspects
3. Clarity: Clear and understandable formulation
4. Relevance: Adequacy with the original query
5. Traceability: References to ISO clauses
6. Actionability: Practical audit questions

IMPORTANT: Respond with EXACTLY 6 numbers from 6-8, separated by commas.
Example: 7,7,8,7,6,7

Your scores:"""
    
    eval_response = llm.generate(evaluation_prompt)
    
    # Extraction des scores
    numbers = re.findall(r'\b([1-9]|10)\b', eval_response)
    scores = [int(n) for n in numbers[:6]] if len(numbers) >= 6 else [7, 7, 7, 7, 7, 7]
    
    return scores

criteria_names = [
    'Conformite ISO',
    'Exhaustivite', 
    'Clarte',
    'Pertinence',
    'Tracabilite',
    'Actionabilite'
]

print("Fonction d'evaluation LLM-as-a-Judge definie")

Fonction d'evaluation LLM-as-a-Judge definie


### 6.2 Evaluation des Resultats

In [12]:
scores_vectoriel = []
scores_graph = []

print("Evaluation LLM-as-a-Judge en cours...")
print("="*70)

for i in range(len(test_queries)):
    query = test_queries[i]
    print(f"\n[{i+1}/5] Evaluation: {query[:40]}...")
    
    # Evaluation methode vectorielle
    scores_v = llm_judge_evaluate(query, results_vectoriel[i]['response'], llm)
    scores_vectoriel.append(scores_v)
    
    # Evaluation methode graph
    scores_g = llm_judge_evaluate(query, results_graph[i]['response'], llm)
    scores_graph.append(scores_g)
    
    print(f"    Vectoriel: {scores_v} (moyenne: {np.mean(scores_v):.1f})")
    print(f"    Graph:     {scores_g} (moyenne: {np.mean(scores_g):.1f})")

print("\n" + "="*70)
print("Evaluation terminee")

Evaluation LLM-as-a-Judge en cours...

[1/5] Evaluation: Data backup and disaster recovery proced...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


    Vectoriel: [10, 9, 10, 10, 9, 10] (moyenne: 9.7)
    Graph:     [10, 9, 10, 10, 8, 9] (moyenne: 9.3)

[2/5] Evaluation: Access control and user authentication p...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


...

INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


    Vectoriel: [10, 9, 10, 10, 9, 10] (moyenne: 9.7)
    Graph:     [10, 9, 10, 10, 9, 9] (moyenne: 9.5)

[5/5] Evaluation: Risk assessment and compliance monitorin...


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"


    Vectoriel: [9, 10, 9, 10, 9, 10] (moyenne: 9.5)
    Graph:     [9, 10, 9, 10, 8, 9] (moyenne: 9.2)

Evaluation terminee
